In [1]:
from data_loader import load_mimic_tables                                      
                                                                                 
tables = load_mimic_tables()                                                                                                        
                                                                                 
# See all available tables:                                                    
print(tables.keys())  

/Users/huseyin/Codes/gpt-medic/explore/data_loader.py:26: DtypeWarning: Columns (4,6,7,8,9,10,11,12,13,15,16,17,18,21,23,24,25,26,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  tables[table_name] = pd.read_csv(file, compression="gzip")


dict_keys(['hosp.poe', 'hosp.d_hcpcs', 'hosp.poe_detail', 'hosp.patients', 'hosp.diagnoses_icd', 'hosp.emar_detail', 'hosp.provider', 'hosp.prescriptions', 'hosp.drgcodes', 'hosp.d_icd_diagnoses', 'hosp.d_labitems', 'hosp.transfers', 'hosp.admissions', 'hosp.labevents', 'hosp.pharmacy', 'hosp.procedures_icd', 'hosp.hcpcsevents', 'hosp.services', 'hosp.d_icd_procedures', 'hosp.omr', 'hosp.emar', 'hosp.microbiologyevents', 'icu.datetimeevents', 'icu.caregiver', 'icu.ingredientevents', 'icu.inputevents', 'icu.procedureevents', 'icu.d_items', 'icu.chartevents', 'icu.icustays', 'icu.outputevents'])


In [2]:
import pandas as pd
from tokenizer.medication import MedicationTokenizer

gsn_atc = pd.read_csv('../data/gsn_atc_ndc_mapping.csv', dtype=str)

prescriptions = tables['hosp.prescriptions'].copy()
prescriptions['gsn'] = prescriptions['gsn'].astype(str).str.zfill(6)

# Drop rows with no GSN
prescriptions = prescriptions[prescriptions['gsn'] != '000nan']

# Map GSN → ATC
prescriptions = prescriptions.merge(
    gsn_atc[['gsn', 'atc']].drop_duplicates('gsn'),
    on='gsn',
    how='left'
).rename(columns={'atc': 'atc_code'})

print(f"Rows after drop:  {len(prescriptions)}")
print(f"ATC mapped:       {prescriptions['atc_code'].notna().sum()}")

med_tokenizer = MedicationTokenizer()
vocab = med_tokenizer.build_vocabulary(prescriptions.dropna(subset=['atc_code']))

print(f"Vocabulary size:  {len(vocab)}")
vocab

Rows after drop:  15568
ATC mapped:       15563
Vocabulary size:  190


{'ATC_4_A': 0,
 'ATC_4_B': 1,
 'ATC_4_C': 2,
 'ATC_4_D': 3,
 'ATC_4_E': 4,
 'ATC_4_F': 5,
 'ATC_4_G': 6,
 'ATC_4_H': 7,
 'ATC_4_M': 8,
 'ATC_4_X': 9,
 'ATC_A01': 10,
 'ATC_A02': 11,
 'ATC_A03': 12,
 'ATC_A04': 13,
 'ATC_A05': 14,
 'ATC_A06': 15,
 'ATC_A07': 16,
 'ATC_A09': 17,
 'ATC_A10': 18,
 'ATC_A11': 19,
 'ATC_A12': 20,
 'ATC_A16': 21,
 'ATC_B01': 22,
 'ATC_B02': 23,
 'ATC_B03': 24,
 'ATC_B05': 25,
 'ATC_C01': 26,
 'ATC_C02': 27,
 'ATC_C03': 28,
 'ATC_C04': 29,
 'ATC_C05': 30,
 'ATC_C07': 31,
 'ATC_C08': 32,
 'ATC_C09': 33,
 'ATC_C10': 34,
 'ATC_D01': 35,
 'ATC_D02': 36,
 'ATC_D03': 37,
 'ATC_D04': 38,
 'ATC_D06': 39,
 'ATC_D07': 40,
 'ATC_D10': 41,
 'ATC_D11': 42,
 'ATC_G02': 43,
 'ATC_G03': 44,
 'ATC_G04': 45,
 'ATC_H01': 46,
 'ATC_H02': 47,
 'ATC_H03': 48,
 'ATC_H04': 49,
 'ATC_H05': 50,
 'ATC_J01': 51,
 'ATC_J02': 52,
 'ATC_J05': 53,
 'ATC_J07': 54,
 'ATC_L01': 55,
 'ATC_L02': 56,
 'ATC_L03': 57,
 'ATC_M01': 58,
 'ATC_M02': 59,
 'ATC_M03': 60,
 'ATC_M04': 61,
 'ATC_M05': 62,
 '

In [3]:
tables['hosp.d_labitems'].head(10)

,itemid,label,fluid,category
0,50808,Free Calcium,Blood,Blood Gas
1,50826,Tidal Volume,Blood,Blood Gas
2,50813,Lactate,Blood,Blood Gas
3,52029,% Ionized Calcium,Blood,Blood Gas
4,50801,Alveolar-arterial Gradient,Blood,Blood Gas
5,50810,"Hematocrit, Calculated",Blood,Blood Gas
6,52025,Delete,Blood,Blood Gas
7,52031,Osmolality,Blood,Blood Gas
8,52022,"Albumin, Blood",Blood,Blood Gas
9,50806,"Chloride, Whole Blood",Blood,Blood Gas


In [4]:
from tokenizer.lab import LabTokenizer

lab_tokenizer = LabTokenizer(n_quantiles=10)
vocab = lab_tokenizer.build_vocabulary(tables['hosp.d_labitems'], tables['hosp.labevents'])

print(f"Vocabulary size: {len(vocab)}")

# Tokenize labevents
labevents_tokenized = lab_tokenizer.tokenize(tables['hosp.labevents'])
print()
labevents_tokenized[['itemid', 'valuenum', 'tokenized_version']].head(20)

Vocabulary size: 1180



,itemid,valuenum,tokenized_version
0,51277,15.40,<LAB RDW> <Q6>
1,51279,3.35,<LAB Red Blood Cells> <Q6>
2,52172,49.70,<LAB RDW-SD> <Q5>
3,51301,20.30,<LAB White Blood Cells> <Q9>
4,51249,31.10,<LAB MCHC> <Q2>
5,51221,29.60,<LAB Hematocrit> <Q5>
6,51222,9.20,<LAB Hemoglobin> <Q5>
7,51265,206.00,<LAB Platelet Count> <Q6>
8,51250,88.00,<LAB MCV> <Q4>
9,51248,27.50,<LAB MCH> <Q3>


In [5]:
print("=== gender ===")
print(tables['hosp.patients']['gender'].unique())
print()
print("=== marital_status ===")
print(tables['hosp.admissions']['marital_status'].unique())
print()
print("=== race ===")
print(tables['hosp.admissions']['race'].unique())

=== gender ===
['F' 'M']

=== marital_status ===
['SINGLE' 'MARRIED' nan 'WIDOWED' 'DIVORCED']

=== race ===
['BLACK/CAPE VERDEAN' 'HISPANIC/LATINO - PUERTO RICAN' 'WHITE' 'UNKNOWN'
 'OTHER' 'BLACK/AFRICAN AMERICAN' 'HISPANIC/LATINO - SALVADORAN'
 'UNABLE TO OBTAIN' 'WHITE - OTHER EUROPEAN' 'PORTUGUESE'
 'HISPANIC/LATINO - CUBAN' 'PATIENT DECLINED TO ANSWER'
 'WHITE - BRAZILIAN' 'HISPANIC OR LATINO']


In [6]:
from tokenizer.demography import DemographyTokenizer

demo_tokenizer = DemographyTokenizer()
vocab = demo_tokenizer.build_vocabulary(tables['hosp.patients'], tables['hosp.admissions'])

print(f"Vocabulary size: {len(vocab)}")
vocab

Vocabulary size: 20


{'MARITAL_DIVORCED': 0,
 'MARITAL_MARRIED': 1,
 'MARITAL_SINGLE': 2,
 'MARITAL_WIDOWED': 3,
 'RACE_BLACK_AFRICAN_AMERICAN': 4,
 'RACE_BLACK_CAPE_VERDEAN': 5,
 'RACE_HISPANIC_LATINO___CUBAN': 6,
 'RACE_HISPANIC_LATINO___PUERTO_RICAN': 7,
 'RACE_HISPANIC_LATINO___SALVADORAN': 8,
 'RACE_HISPANIC_OR_LATINO': 9,
 'RACE_OTHER': 10,
 'RACE_PATIENT_DECLINED_TO_ANSWER': 11,
 'RACE_PORTUGUESE': 12,
 'RACE_UNABLE_TO_OBTAIN': 13,
 'RACE_UNKNOWN': 14,
 'RACE_WHITE': 15,
 'RACE_WHITE___BRAZILIAN': 16,
 'RACE_WHITE___OTHER_EUROPEAN': 17,
 'SEX_F': 18,
 'SEX_M': 19}

In [7]:
from tokenizer.blood_pressure import BloodPressureTokenizer

bp_tokenizer = BloodPressureTokenizer(n_quantiles=10)
vocab = bp_tokenizer.build_vocabulary(tables['hosp.omr'])

print(f"Vocabulary size: {len(vocab)}")
print(vocab)

result = bp_tokenizer.tokenize(tables['hosp.omr'])
result[['result_value', 'systolic', 'diastolic', 'systolic_token', 'diastolic_token', 'tokenized_version']].head(10)

Vocabulary size: 11
{'<Blood_Pressure>': 0, 'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4, 'Q5': 5, 'Q6': 6, 'Q7': 7, 'Q8': 8, 'Q9': 9, 'Q10': 10}


,result_value,systolic,diastolic,systolic_token,diastolic_token,tokenized_version
14,110/70,110,70,Q2,Q6,<Blood_Pressure> Q2 Q6
15,112/80,112,80,Q3,Q9,<Blood_Pressure> Q3 Q9
16,128/84,128,84,Q6,Q10,<Blood_Pressure> Q6 Q10
17,132/84,132,84,Q7,Q10,<Blood_Pressure> Q7 Q10
18,138/90,138,90,Q8,Q10,<Blood_Pressure> Q8 Q10
19,142/90,142,90,Q9,Q10,<Blood_Pressure> Q9 Q10
40,100/60,100,60,Q1,Q3,<Blood_Pressure> Q1 Q3
41,110/70,110,70,Q2,Q6,<Blood_Pressure> Q2 Q6
42,114/70,114,70,Q3,Q6,<Blood_Pressure> Q3 Q6
43,116/70,116,70,Q4,Q6,<Blood_Pressure> Q4 Q6


In [8]:
# ============================================================================
# COMPLETE PATIENT HEALTH TIMELINE TOKENIZATION
# ============================================================================
# This section demonstrates the full timeline tokenizer that combines all
# individual tokenizers and creates a chronological sequence of patient events
# with time interval tokens for gaps >= 5 minutes.
# ============================================================================

import pandas as pd
from tokenizer.timeline import PatientTimelineTokenizer
from tokenizer.time_interval import TimeIntervalTokenizer

# First, let's look at available admissions to pick one
admissions = tables['hosp.admissions']
print("Sample admissions:")
print(admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime']].head(10))

Sample admissions:
   subject_id   hadm_id            admittime            dischtime
0    10004235  24181354  2196-02-24 14:38:00  2196-03-04 14:02:00
1    10009628  25926192  2153-09-17 17:08:00  2153-09-25 13:20:00
2    10018081  23983182  2134-08-18 02:02:00  2134-08-23 19:35:00
3    10006053  22942076  2111-11-13 23:39:00  2111-11-15 17:20:00
4    10031404  21606243  2113-08-04 18:46:00  2113-08-06 20:57:00
5    10005817  20626031  2132-12-12 01:43:00  2132-12-20 15:04:00
6    10019385  20297618  2180-02-15 20:28:00  2180-02-25 13:45:00
7    10002495  24982426  2141-05-22 20:17:00  2141-05-29 17:41:00
8    10038081  20755971  2115-09-27 20:40:00  2115-10-12 00:00:00
9    10019917  22585261  2182-01-07 23:25:00  2182-01-10 16:52:00


In [9]:
# Load GSN -> ATC mapping for medication tokenization
gsn_atc = pd.read_csv('../data/gsn_atc_ndc_mapping.csv', dtype=str)
gsn_to_atc = dict(zip(gsn_atc['gsn'].str.zfill(6), gsn_atc['atc']))
print(f"Loaded {len(gsn_to_atc)} GSN -> ATC mappings")

Loaded 4770 GSN -> ATC mappings


In [10]:
# Initialize and fit the PatientTimelineTokenizer
timeline_tokenizer = PatientTimelineTokenizer(
    n_quantiles=10,
    gsn_to_atc=gsn_to_atc,
)

# Fit all tokenizers on the full dataset
timeline_tokenizer.fit(tables)
print("Timeline tokenizer fitted!")

# Get combined vocabulary size
combined_vocab = timeline_tokenizer.get_combined_vocabulary()
print(f"Combined vocabulary size: {len(combined_vocab)}")

Timeline tokenizer fitted!
Combined vocabulary size: 2265


In [11]:
# Pick a session with good amount of data
# Let's find an admission with many lab events and prescriptions

labevents = tables['hosp.labevents']
prescriptions = tables['hosp.prescriptions']

# Count events per admission
lab_counts = labevents.groupby('hadm_id').size().rename('lab_count')
med_counts = prescriptions.groupby('hadm_id').size().rename('med_count')

admission_stats = admissions[['hadm_id', 'subject_id', 'admittime', 'dischtime']].copy()
admission_stats = admission_stats.merge(lab_counts, on='hadm_id', how='left')
admission_stats = admission_stats.merge(med_counts, on='hadm_id', how='left')
admission_stats = admission_stats.fillna(0)
admission_stats['total_events'] = admission_stats['lab_count'] + admission_stats['med_count']
admission_stats = admission_stats.sort_values('total_events', ascending=False)

print("Top 10 admissions by event count:")
print(admission_stats.head(10))

Top 10 admissions by event count:
      hadm_id  subject_id            admittime            dischtime  \
102  28258130    10039708  2140-01-23 16:19:00  2140-02-26 18:15:00   
216  26486158    10014354  2148-08-22 15:18:00  2148-09-08 12:00:00   
155  22987108    10007818  2146-06-10 16:37:00  2146-07-12 00:00:00   
227  29276678    10035631  2116-02-27 20:55:00  2116-03-12 07:45:00   
226  21476294    10035631  2115-11-08 13:54:00  2115-12-08 17:31:00   
79   28998349    10021487  2116-12-03 00:23:00  2116-12-28 13:19:00   
13   23831430    10020740  2150-03-11 15:34:00  2150-04-25 13:50:00   
26   23559586    10003400  2137-08-04 00:07:00  2137-09-02 17:05:00   
163  29462354    10035631  2112-09-17 19:13:00  2112-10-17 01:41:00   
58   21027282    10018081  2133-12-18 16:58:00  2134-01-12 11:00:00   

     lab_count  med_count  total_events  
102     2390.0      314.0        2704.0  
216     2538.0      116.0        2654.0  
155     2257.0      320.0        2577.0  
227     1869.0  

In [12]:
# Pick the admission with most events and tokenize it
selected_hadm_id = admission_stats.iloc[0]['hadm_id']
print(f"Selected admission: {selected_hadm_id}")
print()

# Tokenize the full session
result = timeline_tokenizer.tokenize_session(int(selected_hadm_id), tables)

print(f"Admission ID:     {result['hadm_id']}")
print(f"Subject ID:       {result['subject_id']}")
print(f"Admit Time:       {result['admit_time']}")
print(f"Discharge Time:   {result['discharge_time']}")
print(f"Total Events:     {len(result['events'])}")
print(f"Total Tokens:     {len(result['timeline_tokens'])}")

Selected admission: 28258130

Admission ID:     28258130
Subject ID:       10039708
Admit Time:       2140-01-23 16:19:00
Discharge Time:   2140-02-26 18:15:00
Total Events:     2686
Total Tokens:     2989


In [13]:
# Show demography tokens (at the start of the timeline)
print("DEMOGRAPHY TOKENS:")
print(result.get('demography_tokens', 'N/A'))
print()

# Show event type breakdown
from collections import Counter
event_types = Counter(e.event_type for e in result['events'])
print("EVENT TYPE BREAKDOWN:")
for etype, count in sorted(event_types.items()):
    print(f"  {etype}: {count}")

DEMOGRAPHY TOKENS:
SEX_F MARITAL_SINGLE RACE_BLACK_AFRICAN_AMERICAN

EVENT TYPE BREAKDOWN:
  DIAG: 38
  LAB: 2390
  MED: 237
  PROC: 21


In [14]:
# Show time interval token distribution
time_interval_tokens = [t for t in result['timeline_tokens'] if t.startswith('<TIME_')]
time_interval_counts = Counter(time_interval_tokens)

print("TIME INTERVAL TOKENS IN THIS SESSION:")
print(f"Total time interval tokens: {len(time_interval_tokens)}")
print()
for token, count in sorted(time_interval_counts.items()):
    print(f"  {token}: {count}")

TIME INTERVAL TOKENS IN THIS SESSION:
Total time interval tokens: 302

  <TIME_12h-1d>: 8
  <TIME_15m-1h>: 59
  <TIME_1h-2h>: 65
  <TIME_2h-6h>: 102
  <TIME_5m-15m>: 41
  <TIME_6h-12h>: 27


In [15]:
# Print the first N events with time intervals (chronological view)
print("=" * 100)
print("CHRONOLOGICAL TIMELINE (first 50 events)")
print("=" * 100)
print()

events = result['events'][:50]
prev_timestamp = None

for event in events:
    # Show time gap if >= 5 minutes
    if prev_timestamp is not None:
        gap = event.timestamp - prev_timestamp
        interval_token = timeline_tokenizer.time_interval_tokenizer.tokenize_gap(gap)
        if interval_token:
            gap_minutes = gap.total_seconds() / 60
            print(f"    >>> {interval_token}  (gap: {gap_minutes:.1f} min)")
    
    # Show event
    print(f"[{event.timestamp}] [{event.event_type:4}] {event.tokens}")
    prev_timestamp = event.timestamp

CHRONOLOGICAL TIMELINE (first 50 events)

[2140-01-23 00:00:00] [PROC] <ICD_PCS_B> <ICD_PCS_5> <ICD_PCS_4> <ICD_PCS_B> <ICD_PCS_Z> <ICD_PCS_Z> <ICD_PCS_A>
[2140-01-23 00:00:00] [PROC] <ICD_PCS_0> <ICD_PCS_6> <ICD_PCS_H> <ICD_PCS_M> <ICD_PCS_3> <ICD_PCS_3> <ICD_PCS_Z>
    >>> <TIME_12h-1d>  (gap: 979.0 min)
[2140-01-23 16:19:00] [DIAG] <ICD_F_D64> <ICD_3_5_9>
[2140-01-23 16:19:00] [DIAG] <ICD_F_N17> <ICD_3_5_0>
[2140-01-23 16:19:00] [DIAG] <ICD_F_J96> <ICD_3_5_91>
[2140-01-23 16:19:00] [DIAG] <ICD_F_I21> <ICD_3_5_4>
[2140-01-23 16:19:00] [DIAG] <ICD_F_J69> <ICD_3_5_0>
[2140-01-23 16:19:00] [DIAG] <ICD_F_G93> <ICD_3_5_40>
[2140-01-23 16:19:00] [DIAG] <ICD_F_I50> <ICD_3_5_21>
[2140-01-23 16:19:00] [DIAG] <ICD_F_J95> <ICD_3_5_85> <ICD_6_1>
[2140-01-23 16:19:00] [DIAG] <ICD_F_E43>
[2140-01-23 16:19:00] [DIAG] <ICD_F_R64>
[2140-01-23 16:19:00] [DIAG] <ICD_F_E87> <ICD_3_5_2>
[2140-01-23 16:19:00] [DIAG] <ICD_F_E51> <ICD_3_5_12>
[2140-01-23 16:19:00] [DIAG] <ICD_F_I82> <ICD_3_5_40> <ICD_6_1>
[

In [16]:
# Show the full token sequence (first 2000 chars)
print("=" * 100)
print("FULL TOKEN SEQUENCE (first 2000 chars)")
print("=" * 100)
print()

full_seq = result['full_sequence']
print(f"Total sequence length: {len(full_seq)} characters")
print()
print(full_seq[:2000])
if len(full_seq) > 2000:
    print("...")
    print(f"\n[Truncated - full sequence is {len(full_seq)} characters]")

FULL TOKEN SEQUENCE (first 2000 chars)

Total sequence length: 66186 characters

SEX_F MARITAL_SINGLE RACE_BLACK_AFRICAN_AMERICAN <ICD_PCS_B> <ICD_PCS_5> <ICD_PCS_4> <ICD_PCS_B> <ICD_PCS_Z> <ICD_PCS_Z> <ICD_PCS_A> <ICD_PCS_0> <ICD_PCS_6> <ICD_PCS_H> <ICD_PCS_M> <ICD_PCS_3> <ICD_PCS_3> <ICD_PCS_Z> <TIME_12h-1d> <ICD_F_D64> <ICD_3_5_9> <ICD_F_N17> <ICD_3_5_0> <ICD_F_J96> <ICD_3_5_91> <ICD_F_I21> <ICD_3_5_4> <ICD_F_J69> <ICD_3_5_0> <ICD_F_G93> <ICD_3_5_40> <ICD_F_I50> <ICD_3_5_21> <ICD_F_J95> <ICD_3_5_85> <ICD_6_1> <ICD_F_E43> <ICD_F_R64> <ICD_F_E87> <ICD_3_5_2> <ICD_F_E51> <ICD_3_5_12> <ICD_F_I82> <ICD_3_5_40> <ICD_6_1> <ICD_F_Z68> <ICD_3_5_1> <ICD_F_E87> <ICD_3_5_3> <ICD_F_I51> <ICD_3_5_81> <ICD_F_E87> <ICD_3_5_0> <ICD_F_R34> <ICD_F_D61> <ICD_3_5_81> <ICD_6_8> <ICD_F_D69> <ICD_3_5_6> <ICD_F_R68> <ICD_3_5_0> <ICD_F_F10> <ICD_3_5_20> <ICD_F_F17> <ICD_3_5_21> <ICD_6_0> <ICD_F_E03> <ICD_3_5_9> <ICD_F_I10> <ICD_F_I34> <ICD_3_5_0> <ICD_F_K70> <ICD_3_5_30> <ICD_F_K70> <ICD_3_5_10> <ICD_F_E59> 

In [17]:
# Show the time interval vocabulary
print("TIME INTERVAL VOCABULARY:")
print()
for token, idx in timeline_tokenizer.time_interval_tokenizer.vocabulary.items():
    print(f"  {idx:2}: {token}")

TIME INTERVAL VOCABULARY:

   0: <TIME_5m-15m>
   1: <TIME_15m-1h>
   2: <TIME_1h-2h>
   3: <TIME_2h-6h>
   4: <TIME_6h-12h>
   5: <TIME_12h-1d>
   6: <TIME_1d-3d>
   7: <TIME_3d-1w>
   8: <TIME_1w-2w>
   9: <TIME_2w-1mt>
  10: <TIME_1mt-3mt>
  11: <TIME_3mt-6mt>
  12: <TIME_>=6mt>


In [18]:
# Show individual tokens as they would be fed to a model
print("=" * 100)
print("INDIVIDUAL TOKENS (first 100)")
print("=" * 100)
print()

# Split full sequence into individual tokens
all_individual_tokens = result['full_sequence'].split()
print(f"Total individual tokens: {len(all_individual_tokens)}")
print()

for i, token in enumerate(all_individual_tokens[:100]):
    print(f"[{i:4}] {token}")

INDIVIDUAL TOKENS (first 100)

Total individual tokens: 9447

[   0] SEX_F
[   1] MARITAL_SINGLE
[   2] RACE_BLACK_AFRICAN_AMERICAN
[   3] <ICD_PCS_B>
[   4] <ICD_PCS_5>
[   5] <ICD_PCS_4>
[   6] <ICD_PCS_B>
[   7] <ICD_PCS_Z>
[   8] <ICD_PCS_Z>
[   9] <ICD_PCS_A>
[  10] <ICD_PCS_0>
[  11] <ICD_PCS_6>
[  12] <ICD_PCS_H>
[  13] <ICD_PCS_M>
[  14] <ICD_PCS_3>
[  15] <ICD_PCS_3>
[  16] <ICD_PCS_Z>
[  17] <TIME_12h-1d>
[  18] <ICD_F_D64>
[  19] <ICD_3_5_9>
[  20] <ICD_F_N17>
[  21] <ICD_3_5_0>
[  22] <ICD_F_J96>
[  23] <ICD_3_5_91>
[  24] <ICD_F_I21>
[  25] <ICD_3_5_4>
[  26] <ICD_F_J69>
[  27] <ICD_3_5_0>
[  28] <ICD_F_G93>
[  29] <ICD_3_5_40>
[  30] <ICD_F_I50>
[  31] <ICD_3_5_21>
[  32] <ICD_F_J95>
[  33] <ICD_3_5_85>
[  34] <ICD_6_1>
[  35] <ICD_F_E43>
[  36] <ICD_F_R64>
[  37] <ICD_F_E87>
[  38] <ICD_3_5_2>
[  39] <ICD_F_E51>
[  40] <ICD_3_5_12>
[  41] <ICD_F_I82>
[  42] <ICD_3_5_40>
[  43] <ICD_6_1>
[  44] <ICD_F_Z68>
[  45] <ICD_3_5_1>
[  46] <ICD_F_E87>
[  47] <ICD_3_5_3>
[  48] <I